# 缺失值插补优化版

## 模型说明
- **模型1-3**：Linear/Ridge/Lasso → RidgeCV/LassoCV（无需Optuna）
- **模型4**：KNNR → Optuna TPE n_trials=50
- **模型5**：SVR → Optuna TPE n_trials=50
- **模型6**：DTR → Optuna TPE n_trials=20（搜索空间小）
- **模型7**：GBR → Optuna TPE n_trials=50
- **模型8**：RFR → Optuna TPE n_trials=50
- **模型9**：ETR → Optuna TPE n_trials=50
- **模型10**：XGBR → Optuna TPE n_trials=50（CV用CPU，最终训练GPU）
- **模型11**：DNNR_rect → Optuna TPE n_trials=50 + HyperbandPruner（GPU）
- **模型12**：DNNR_rand → Optuna TPE n_trials=50 + HyperbandPruner（GPU）

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (GradientBoostingRegressor,
                               RandomForestRegressor,
                               ExtraTreesRegressor)
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy import stats

optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}")

# ── 评估函数 ──────────────────────────────────────────────────
def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    return {
        'MAE':  round(float(mean_absolute_error(y_test, pred)), 4),
        'RMSE': round(float(np.sqrt(mean_squared_error(y_test, pred))), 4),
        'R2':   round(float(r2_score(y_test, pred)), 4)
    }

# ── PyTorch DNN ───────────────────────────────────────────────
class DNN(nn.Module):
    def __init__(self, in_dim, hidden_sizes, activation):
        super().__init__()
        act_map = {'relu': nn.ReLU(), 'tanh': nn.Tanh(), 'sigmoid': nn.Sigmoid()}
        layers, d = [], in_dim
        for h in hidden_sizes:
            layers += [nn.Linear(d, h), act_map[activation]]
            d = h
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)

def train_dnn(model, X, y, optimizer, batch_size, epochs, trial=None):
    model.train()
    Xt = torch.FloatTensor(X).to(device)
    yt = torch.FloatTensor(y).to(device)
    nv = max(1, int(len(Xt) * 0.1))
    Xv, yv, Xt, yt = Xt[-nv:], yt[-nv:], Xt[:-nv], yt[:-nv]
    crit = nn.L1Loss()
    for epoch in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i+batch_size]
            optimizer.zero_grad()
            crit(model(Xt[idx]), yt[idx]).backward()
            optimizer.step()
        if trial is not None and epoch % 10 == 0:
            with torch.no_grad():
                val_mae = crit(model(Xv), yv).item()
            trial.report(val_mae, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

def predict_dnn(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.FloatTensor(X).to(device)).cpu().numpy()

class TorchWrapper:
    def __init__(self, m): 
        self.model = m
    def predict(self, X):  
        return predict_dnn(self.model, X)

# ══════════════════════════════════════════════════════════════
# 核心函数：12个模型竞争
# 输入：X_tr/y_tr 训练集，X_te/y_te 封存测试集，step_name 标识
# 输出：results, df_res, best_name, best_model, best_scaled, scaler
# ══════════════════════════════════════════════════════════════
def run_competition(X_tr, y_tr, X_te, y_te, step_name):

    results   = {}
    input_dim = X_tr.shape[1]
    scaler    = StandardScaler()
    X_tr_sc   = scaler.fit_transform(X_tr)
    X_te_sc   = scaler.transform(X_te)

    # 1. Linear
    m = LinearRegression().fit(X_tr_sc, y_tr)
    results['Linear'] = evaluate_model(m, X_te_sc, y_te)
    results['Linear'].update({'model': m, 'scaled': True})
    print(f"[{step_name}] Linear   MAE={results['Linear']['MAE']}")

    # 2. Ridge
    m = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(X_tr_sc, y_tr)
    results['Ridge'] = evaluate_model(m, X_te_sc, y_te)
    results['Ridge'].update({'model': m, 'scaled': True})
    print(f"[{step_name}] Ridge    MAE={results['Ridge']['MAE']},  alpha={m.alpha_:.4f}")

    # 3. Lasso
    m = LassoCV(cv=5, max_iter=10000, tol=1e-4).fit(X_tr_sc, y_tr)
    results['Lasso'] = evaluate_model(m, X_te_sc, y_te)
    results['Lasso'].update({'model': m, 'scaled': True})
    print(f"[{step_name}] Lasso    MAE={results['Lasso']['MAE']},  alpha={m.alpha_:.4f}")

    # 4. KNNR
    def obj_knn(trial):
        al = trial.suggest_categorical('algorithm', ['kd_tree','brute','ball_tree'])
        m  = Pipeline([('sc', StandardScaler()),
                       ('m', KNeighborsRegressor(
                           n_neighbors=trial.suggest_int('n_neighbors', 1, 20),
                           algorithm=al,
                           leaf_size=trial.suggest_int('leaf_size', 10, 50) if al != 'brute' else 30,
                           p=trial.suggest_int('p', 1, 5)))])
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_knn, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = Pipeline([('sc', StandardScaler()),
                   ('m', KNeighborsRegressor(
                       n_neighbors=bp['n_neighbors'],
                       algorithm=bp['algorithm'],
                       leaf_size=bp.get('leaf_size', 30),
                       p=bp['p']))]).fit(X_tr, y_tr)
    results['KNNR'] = evaluate_model(m, X_te, y_te)
    results['KNNR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] KNNR     MAE={results['KNNR']['MAE']},  best={bp}")

    # 5. SVR
    def obj_svr(trial):
        m = Pipeline([('sc', StandardScaler()),
                      ('m', SVR(
                          kernel=trial.suggest_categorical('kernel', ['rbf','poly','sigmoid']),
                          gamma=trial.suggest_float('gamma', 1e-3, 1.0, log=True),
                          C=trial.suggest_float('C', 1.0, 100.0, log=True)))])
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_svr, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = Pipeline([('sc', StandardScaler()), ('m', SVR(**bp))]).fit(X_tr, y_tr)
    results['SVR'] = evaluate_model(m, X_te, y_te)
    results['SVR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] SVR      MAE={results['SVR']['MAE']},  best={bp}")

    # 6. DTR（n_trials=20）
    def obj_dtr(trial):
        m = DecisionTreeRegressor(
            criterion=trial.suggest_categorical('criterion', ['absolute_error','squared_error']),
            splitter=trial.suggest_categorical('splitter', ['best','random']),
            max_depth=trial.suggest_int('max_depth', 3, 50), random_state=42)
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_dtr, n_trials=20, show_progress_bar=False)
    bp = study.best_params
    m  = DecisionTreeRegressor(**bp, random_state=42).fit(X_tr, y_tr)
    results['DTR'] = evaluate_model(m, X_te, y_te)
    results['DTR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] DTR      MAE={results['DTR']['MAE']},  best={bp}")

    # 7. GBR
    def obj_gbr(trial):
        m = GradientBoostingRegressor(
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
            loss=trial.suggest_categorical('loss', ['absolute_error','huber']),
            n_estimators=trial.suggest_int('n_estimators', 10, 200),
            max_depth=trial.suggest_int('max_depth', 1, 20),
            alpha=trial.suggest_float('alpha', 0.01, 0.99), random_state=42)
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_gbr, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = GradientBoostingRegressor(**bp, random_state=42).fit(X_tr, y_tr)
    results['GBR'] = evaluate_model(m, X_te, y_te)
    results['GBR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] GBR      MAE={results['GBR']['MAE']},  best={bp}")

    # 8. RFR
    def obj_rfr(trial):
        m = RandomForestRegressor(
            n_estimators=trial.suggest_int('n_estimators', 10, 500),
            max_depth=trial.suggest_int('max_depth', 3, 50), random_state=42)
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_rfr, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = RandomForestRegressor(**bp, random_state=42).fit(X_tr, y_tr)
    results['RFR'] = evaluate_model(m, X_te, y_te)
    results['RFR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] RFR      MAE={results['RFR']['MAE']},  best={bp}")

    # 9. ETR
    def obj_etr(trial):
        m = ExtraTreesRegressor(
            n_estimators=trial.suggest_int('n_estimators', 10, 500),
            max_depth=trial.suggest_int('max_depth', 3, 50), random_state=42)
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_etr, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = ExtraTreesRegressor(**bp, random_state=42).fit(X_tr, y_tr)
    results['ETR'] = evaluate_model(m, X_te, y_te)
    results['ETR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] ETR      MAE={results['ETR']['MAE']},  best={bp}")

    # 10. XGBR（CV用CPU，最终训练GPU）
    def obj_xgbr(trial):
        m = XGBRegressor(
            eta=trial.suggest_float('eta', 0.01, 0.5, log=True),
            n_estimators=trial.suggest_int('n_estimators', 10, 200),
            max_depth=trial.suggest_int('max_depth', 1, 50),
            subsample=trial.suggest_float('subsample', 0.3, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.01, 1.0),
            random_state=42, verbosity=0, device='cpu')
        return -cross_val_score(m, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error').mean()
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(obj_xgbr, n_trials=50, show_progress_bar=False)
    bp = study.best_params
    m  = XGBRegressor(**bp, random_state=42, verbosity=0, device='cuda').fit(X_tr, y_tr)
    results['XGBR'] = evaluate_model(m, X_te, y_te)
    results['XGBR'].update({'model': m, 'scaled': False})
    print(f"[{step_name}] XGBR     MAE={results['XGBR']['MAE']},  best={bp}")

    # 11. DNNR_rectangle（每层单元数相同）
    def obj_rect(trial):
        n_layers   = trial.suggest_int('n_layers', 2, 6)
        units      = trial.suggest_int('units', 32, 256, step=32)
        activation = trial.suggest_categorical('activation', ['relu','tanh','sigmoid'])
        lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        opt_name   = trial.suggest_categorical('optimizer', ['Adam','SGD','RMSprop'])
        batch_size = trial.suggest_int('batch_size', 16, 64, step=8)
        model = DNN(input_dim, [units]*n_layers, activation).to(device)
        opt   = getattr(torch.optim, opt_name)(model.parameters(), lr=lr)
        train_dnn(model, X_tr_sc, y_tr, opt, batch_size, 200, trial)
        return mean_absolute_error(y_tr, predict_dnn(model, X_tr_sc))

    print(f"[{step_name}] 搜索 DNNR_rectangle...")
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.HyperbandPruner(min_resource=10, max_resource=200, reduction_factor=3))
    study.optimize(obj_rect, n_trials=50, show_progress_bar=False)
    bp    = study.best_params
    model = DNN(input_dim, [bp['units']]*bp['n_layers'], bp['activation']).to(device)
    opt   = getattr(torch.optim, bp['optimizer'])(model.parameters(), lr=bp['lr'])
    train_dnn(model, X_tr_sc, y_tr, opt, bp['batch_size'], 200)
    results['DNNR_rect'] = evaluate_model(TorchWrapper(model), X_te_sc, y_te)
    results['DNNR_rect'].update({'model': TorchWrapper(model), 'scaled': True})
    print(f"[{step_name}] DNNR_rect MAE={results['DNNR_rect']['MAE']},  best={bp}")

    # 12. DNNR_random（每层单元数独立搜索）
    def obj_rand(trial):
        n_layers   = trial.suggest_int('n_layers', 2, 6)
        activation = trial.suggest_categorical('activation', ['relu','tanh','sigmoid'])
        lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        opt_name   = trial.suggest_categorical('optimizer', ['Adam','SGD','RMSprop'])
        batch_size = trial.suggest_int('batch_size', 16, 64, step=8)
        hidden = [trial.suggest_int(f'units_{i}', 32, 256, step=32) for i in range(n_layers)]
        model = DNN(input_dim, hidden, activation).to(device)
        opt   = getattr(torch.optim, opt_name)(model.parameters(), lr=lr)
        train_dnn(model, X_tr_sc, y_tr, opt, batch_size, 200, trial)
        return mean_absolute_error(y_tr, predict_dnn(model, X_tr_sc))

    print(f"[{step_name}] 搜索 DNNR_random...")
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.HyperbandPruner(min_resource=10, max_resource=200, reduction_factor=3))
    study.optimize(obj_rand, n_trials=50, show_progress_bar=False)
    bp     = study.best_params
    n_l    = bp['n_layers']
    hidden = [bp[f'units_{i}'] for i in range(n_l)]
    model  = DNN(input_dim, hidden, bp['activation']).to(device)
    opt    = getattr(torch.optim, bp['optimizer'])(model.parameters(), lr=bp['lr'])
    train_dnn(model, X_tr_sc, y_tr, opt, bp['batch_size'], 200)
    results['DNNR_rand'] = evaluate_model(TorchWrapper(model), X_te_sc, y_te)
    results['DNNR_rand'].update({'model': TorchWrapper(model), 'scaled': True})
    print(f"[{step_name}] DNNR_rand MAE={results['DNNR_rand']['MAE']},  best={bp}")

    # ── 汇总排名 ──────────────────────────────────────────────
    df_res = pd.DataFrame({
        k: {'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R2': v['R2']}
        for k, v in results.items()
    }).T.astype(float).sort_values('MAE')

    print(f"\n[{step_name}] 结果汇总：")
    print(df_res.to_string())

    best_name   = df_res.index[0]
    best_model  = results[best_name]['model']
    best_scaled = results[best_name]['scaled']
    print(f"\n[{step_name}] 最优: {best_name}  MAE={df_res.loc[best_name,'MAE']}  scaled={best_scaled}")

    return results, df_res, best_name, best_model, best_scaled, scaler

print("Cell 0 完成，run_competition 函数已定义")

GPU: NVIDIA GeForce RTX 5080
Cell 0 完成，run_competition 函数已定义


Cell 1：加载数据 + 记录原始索引

In [2]:
import numpy as np
import pandas as pd

features = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr',
            'Li','Ni','Be','Sc','Tsol','Tage','tage']
targets  = ['YS', 'UTS', 'El']

df = pd.read_excel('dedup_and_merge.xlsx')
print(f"数据shape: {df.shape}")
print(f"\n缺失情况:")
print(df[targets].isnull().sum())

# 在任何插补前记录原始有实测值的行索引
obs_idx = {
    'UTS': df[df['UTS'].notna()].index,
    'YS': df[df['YS'].notna()].index, 
    'El': df[df['El'].notna()].index
}

for target, idx in obs_idx.items():
    print(f"{target} 原始有值: {len(idx)} 行")

数据shape: (491, 23)

缺失情况:
YS     49
UTS    15
El     37
dtype: int64
UTS 原始有值: 476 行
YS 原始有值: 442 行
El 原始有值: 454 行


Cell 2：阶段1 — features → UTS

In [3]:
# 阶段1 — features → UTS
df_s1 = df.loc[obs_idx['UTS']].copy()
X1, y1 = df_s1[features].values, df_s1['UTS'].values
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, random_state=42)

_, _, best_nm1, best_m1, best_sc1, scaler1 = run_competition(
    X1_tr, y1_tr, X1_te, y1_te, '阶段1_UTS')

# 填补 UTS 缺失
mask_uts = df['UTS'].isna()
print(f"\n需要填补 UTS 的行数: {mask_uts.sum()}")
X_fill = df.loc[mask_uts, features].values
X_fill = scaler1.transform(X_fill) if best_sc1 else X_fill
df.loc[mask_uts, 'UTS'] = best_m1.predict(X_fill)
print(f"UTS 填补完成，剩余缺失: {df['UTS'].isna().sum()}")

[阶段1_UTS] Linear   MAE=44.1614
[阶段1_UTS] Ridge    MAE=45.1699,  alpha=44.9843
[阶段1_UTS] Lasso    MAE=45.7234,  alpha=1.8530
[阶段1_UTS] KNNR     MAE=31.9229,  best={'algorithm': 'kd_tree', 'n_neighbors': 2, 'leaf_size': 38, 'p': 3}
[阶段1_UTS] SVR      MAE=35.8938,  best={'kernel': 'rbf', 'gamma': 0.20455015848536376, 'C': 99.684596584385}
[阶段1_UTS] DTR      MAE=37.6165,  best={'criterion': 'absolute_error', 'splitter': 'random', 'max_depth': 24}
[阶段1_UTS] GBR      MAE=24.8258,  best={'learning_rate': 0.10769622478263129, 'loss': 'absolute_error', 'n_estimators': 191, 'max_depth': 20, 'alpha': 0.8022294011541319}
[阶段1_UTS] RFR      MAE=26.8663,  best={'n_estimators': 209, 'max_depth': 43}
[阶段1_UTS] ETR      MAE=27.0615,  best={'n_estimators': 240, 'max_depth': 16}
[阶段1_UTS] XGBR     MAE=28.2637,  best={'eta': 0.12279231304548048, 'n_estimators': 129, 'max_depth': 41, 'subsample': 0.4236388708355149, 'colsample_bytree': 0.9983007503043428}
[阶段1_UTS] 搜索 DNNR_rectangle...
[阶段1_UTS] DNNR_rect 

Cell 3：阶段2 — features + 实测UTS → YS

In [4]:
# 阶段2 — features + 实测UTS → YS
bridge_idx = obs_idx['UTS'].intersection(obs_idx['YS'])
df_s2 = df.loc[bridge_idx].copy()
X2 = df_s2[features + ['UTS']].values  # 用实测UTS作为特征
y2 = df_s2['YS'].values

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)

_, _, best_nm2, best_m2, best_sc2, scaler2 = run_competition(
    X2_tr, y2_tr, X2_te, y2_te, '阶段2_YS')

# 推理：所有 YS 缺失的行
mask_ys = df['YS'].isna()
print(f"\n需要填补 YS 的行数: {mask_ys.sum()}")

# 此时 df['UTS'] 已全部填满（实测值 or 阶段1预测值）
X_fill = df.loc[mask_ys, features + ['UTS']].values
X_fill = scaler2.transform(X_fill) if best_sc2 else X_fill
df.loc[mask_ys, 'YS'] = best_m2.predict(X_fill)
print(f"YS 填补完成，剩余缺失: {df['YS'].isna().sum()}")

[阶段2_YS] Linear   MAE=24.3218
[阶段2_YS] Ridge    MAE=24.3371,  alpha=0.1600
[阶段2_YS] Lasso    MAE=24.0279,  alpha=0.6202
[阶段2_YS] KNNR     MAE=34.3807,  best={'algorithm': 'ball_tree', 'n_neighbors': 1, 'leaf_size': 36, 'p': 3}
[阶段2_YS] SVR      MAE=28.8709,  best={'kernel': 'poly', 'gamma': 0.1609017833004847, 'C': 16.196165548249482}
[阶段2_YS] DTR      MAE=26.7474,  best={'criterion': 'absolute_error', 'splitter': 'best', 'max_depth': 36}
[阶段2_YS] GBR      MAE=14.9627,  best={'learning_rate': 0.08216962745870568, 'loss': 'absolute_error', 'n_estimators': 157, 'max_depth': 14, 'alpha': 0.5687311407272482}
[阶段2_YS] RFR      MAE=18.0141,  best={'n_estimators': 35, 'max_depth': 29}
[阶段2_YS] ETR      MAE=15.7849,  best={'n_estimators': 497, 'max_depth': 20}
[阶段2_YS] XGBR     MAE=14.8605,  best={'eta': 0.0885391461883862, 'n_estimators': 194, 'max_depth': 49, 'subsample': 0.33370111001086733, 'colsample_bytree': 0.8625649342047428}
[阶段2_YS] 搜索 DNNR_rectangle...
[阶段2_YS] DNNR_rect MAE=26.1253

Cell 4：阶段3 — features + 实测UTS + 实测YS → El

In [5]:
# 阶段3 — features + 实测UTS + 实测YS → El
el_train_idx = obs_idx['UTS'].intersection(obs_idx['YS']).intersection(obs_idx['El'])
df_s3 = df.loc[el_train_idx].copy()
X3 = df_s3[features + ['UTS', 'YS']].values  # 用实测UTS和YS
y3 = df_s3['El'].values

X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2, random_state=42)

_, _, best_nm3, best_m3, best_sc3, scaler3 = run_competition(
    X3_tr, y3_tr, X3_te, y3_te, '阶段3_El')

# 推理：所有 El 缺失的行
mask_el = df['El'].isna()
print(f"\n需要填补 El 的行数: {mask_el.sum()}")

# 此时 UTS 和 YS 已全部填满
X_fill = df.loc[mask_el, features + ['UTS', 'YS']].values
X_fill = scaler3.transform(X_fill) if best_sc3 else X_fill
df.loc[mask_el, 'El'] = best_m3.predict(X_fill)
print(f"El 填补完成，剩余缺失: {df['El'].isna().sum()}")

[阶段3_El] Linear   MAE=3.449
[阶段3_El] Ridge    MAE=3.3398,  alpha=14.5635
[阶段3_El] Lasso    MAE=3.391,  alpha=0.0630
[阶段3_El] KNNR     MAE=2.2443,  best={'algorithm': 'kd_tree', 'n_neighbors': 2, 'leaf_size': 38, 'p': 3}
[阶段3_El] SVR      MAE=2.3634,  best={'kernel': 'rbf', 'gamma': 0.10241804111387831, 'C': 72.39245553911978}
[阶段3_El] DTR      MAE=3.1701,  best={'criterion': 'absolute_error', 'splitter': 'best', 'max_depth': 36}
[阶段3_El] GBR      MAE=2.0317,  best={'learning_rate': 0.10121098823604023, 'loss': 'absolute_error', 'n_estimators': 183, 'max_depth': 19, 'alpha': 0.6847316467588064}
[阶段3_El] RFR      MAE=2.2589,  best={'n_estimators': 88, 'max_depth': 45}
[阶段3_El] ETR      MAE=2.0288,  best={'n_estimators': 403, 'max_depth': 40}
[阶段3_El] XGBR     MAE=1.8519,  best={'eta': 0.03664778904535752, 'n_estimators': 197, 'max_depth': 7, 'subsample': 0.7253165649772657, 'colsample_bytree': 0.32419174389200633}
[阶段3_El] 搜索 DNNR_rectangle...
[阶段3_El] DNNR_rect MAE=2.704,  best={'n_laye

Cell 5：验证  + 导出

In [10]:
# 验证全部填补完成
print("插补后缺失情况（全部应为0）：")
print(df[targets].isnull().sum())
print(f"\n插补后 shape: {df.shape}")

# ══════════════════════════════════════════════════════════════
# 重要：在划分测试集之前，先评估插补效果
# 使用RF默认参数进行100次随机种子实验，做t检验
# ══════════════════════════════════════════════════════════════

def evaluate_rf_imputation_effect(orig_df, imp_df, features, targets, n_seeds=100):
    """
    评估RF模型在插补前后的性能变化
    对比：插补前（去掉缺失样本）vs 插补后（完整数据集）
    """
    results_before, results_after = {}, {}

    for target in targets:
        print(f"\n=== 评估 {target} 插补效果 ===")

        # 插补前：只使用完整样本
        complete_samples = orig_df.dropna(subset=[target])
        X_before, y_before = complete_samples[features].values, complete_samples[target].values
        
        # 插补后：使用完整的数据集
        X_after, y_after = imp_df[features].values, imp_df[target].values

        print(f"  插补前样本数: {len(X_before)}")
        print(f"  插补后样本数: {len(X_after)}")

        # 存储每个种子的性能
        metrics_before = {'MAE': [], 'RMSE': [], 'R2': []}
        metrics_after = {'MAE': [], 'RMSE': [], 'R2': []}

        for seed in range(n_seeds):
            # 插补前：在完整样本上训练和验证
            X_tr_bef, X_te_bef, y_tr_bef, y_te_bef = train_test_split(
                X_before, y_before, test_size=0.2, random_state=seed)

            # 插补后：在完整数据集上训练和验证
            X_tr_aft, X_te_aft, y_tr_aft, y_te_aft = train_test_split(
                X_after, y_after, test_size=0.2, random_state=seed)

            # 训练RF模型（使用默认参数）
            rf = RandomForestRegressor(random_state=seed, n_jobs=-1)

            # 插补前数据训练和评估
            rf.fit(X_tr_bef, y_tr_bef)
            pred_bef = rf.predict(X_te_bef)
            metrics_before['MAE'].append(mean_absolute_error(y_te_bef, pred_bef))
            metrics_before['RMSE'].append(np.sqrt(mean_squared_error(y_te_bef, pred_bef)))
            metrics_before['R2'].append(r2_score(y_te_bef, pred_bef))

            # 插补后数据训练和评估
            rf.fit(X_tr_aft, y_tr_aft)
            pred_aft = rf.predict(X_te_aft)
            metrics_after['MAE'].append(mean_absolute_error(y_te_aft, pred_aft))
            metrics_after['RMSE'].append(np.sqrt(mean_squared_error(y_te_aft, pred_aft)))
            metrics_after['R2'].append(r2_score(y_te_aft, pred_aft))

        # 存储结果
        results_before[target] = metrics_before
        results_after[target] = metrics_after

        # 执行t检验
        def print_stats(before, after, metric):
            t_stat, p_val = stats.ttest_rel(before, after)
            bef_mean, bef_std = np.mean(before), np.std(before)
            aft_mean, aft_std = np.mean(after), np.std(after)
            
            print(f"  插补前: {metric}={bef_mean:.4f} ± {bef_std:.4f}")
            print(f"  插补后: {metric}={aft_mean:.4f} ± {aft_std:.4f}")
            print(f"  统计检验: t={t_stat:.4f}, p={p_val:.4f}")
            
            improved = (metric == 'R2' and aft_mean > bef_mean) or (metric != 'R2' and aft_mean < bef_mean)
            sig = "显著" if p_val < 0.05 else "不显著"
            change = "改善" if improved else "恶化"
            print(f"  → {metric}变化{sig} ({change})")
            
            return p_val, improved

        print(f"{target} 插补效果评估 (n={n_seeds} 次重复):")
        print_stats(metrics_before['MAE'], metrics_after['MAE'], 'MAE')
        print_stats(metrics_before['RMSE'], metrics_after['RMSE'], 'RMSE')
        print_stats(metrics_before['R2'], metrics_after['R2'], 'R2')

    return results_before, results_after

print("\n" + "="*60)
print("开始评估插补效果 - RF模型100次重复实验 + t检验")
print("="*60)

# 保存插补前的原始数据用于对比
df_orig = pd.read_excel('dedup_and_merge.xlsx')

# 执行插补效果评估
results_before, results_after = evaluate_rf_imputation_effect(
    df_orig, df, features, targets, n_seeds=100)

print("\n" + "="*60)
print("插补效果评估完成！")
print("总结：")
for target in targets:
    mae_bef = np.mean(results_before[target]['MAE'])
    mae_aft = np.mean(results_after[target]['MAE'])
    rmse_bef = np.mean(results_before[target]['RMSE'])
    rmse_aft = np.mean(results_after[target]['RMSE'])
    r2_bef = np.mean(results_before[target]['R2'])
    r2_aft = np.mean(results_after[target]['R2'])

    print(f"{target}:")
    print(f"  MAE  {mae_bef:.4f} → {mae_aft:.4f}")
    print(f"  RMSE {rmse_bef:.4f} → {rmse_aft:.4f}")
    print(f"  R²   {r2_bef:.4f} → {r2_aft:.4f}")

print("\n如果插补有效（MAE/RMSE降低且R²提高），可以继续划分测试集")
print("如果插补效果不佳，建议重新考虑插补策略")
print("="*60)



插补后缺失情况（全部应为0）：
YS     0
UTS    0
El     0
dtype: int64

插补后 shape: (491, 23)

开始评估插补效果 - RF模型100次重复实验 + t检验

=== 评估 YS 插补效果 ===
  插补前样本数: 442
  插补后样本数: 491
YS 插补效果评估 (n=100 次重复):
  插补前: MAE=30.9282 ± 3.3301
  插补后: MAE=31.1564 ± 3.1502
  统计检验: t=-0.5066, p=0.6135
  → MAE变化不显著 (恶化)
  插补前: RMSE=43.5606 ± 4.5401
  插补后: RMSE=43.6435 ± 4.4589
  统计检验: t=-0.1294, p=0.8973
  → RMSE变化不显著 (恶化)
  插补前: R2=0.8738 ± 0.0267
  插补后: R2=0.8784 ± 0.0262
  统计检验: t=-1.2168, p=0.2266
  → R2变化不显著 (改善)

=== 评估 UTS 插补效果 ===
  插补前样本数: 476
  插补后样本数: 491
UTS 插补效果评估 (n=100 次重复):
  插补前: MAE=28.2382 ± 2.8649
  插补后: MAE=27.8696 ± 3.1227
  统计检验: t=0.8792, p=0.3814
  → MAE变化不显著 (改善)
  插补前: RMSE=40.7547 ± 4.9287
  插补后: RMSE=39.8024 ± 4.9077
  统计检验: t=1.3886, p=0.1681
  → RMSE变化不显著 (改善)
  插补前: R2=0.8638 ± 0.0392
  插补后: R2=0.8691 ± 0.0369
  统计检验: t=-1.0413, p=0.3003
  → R2变化不显著 (改善)

=== 评估 El 插补效果 ===
  插补前样本数: 454
  插补后样本数: 491
El 插补效果评估 (n=100 次重复):
  插补前: MAE=2.3824 ± 0.3224
  插补后: MAE=2.3010 ± 0.2838
  统计检验: t=1.9672

In [ ]:
# ══════════════════════════════════════════════════════════════
# 导出完整的插补后数据集
# ══════════════════════════════════════════════════════════════
df.to_excel('df_imputed.xlsx', index=False)
print("\n✅ 完整插补数据集已导出: df_imputed.xlsx")